# 10 機器學習：預測感染與重症

能否從住民的基本資料預測誰會感染、誰會變重症？

流程：**問題定義 → 特徵工程 → Pipeline → 交叉驗證 AUC → Random Forest → 特徵重要性 → 模型比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — 問題定義：把長官的問題變成 0/1 標籤

ML 不吃「預測誰會生病」這種句子，它需要一欄明確的 **0/1**。我們定義 `infected`（是否感染，43% 正例）和 `severe_outcome`（住院或死亡，24% 正例）。

> 💡 **標籤一定了，「作弊欄位」也定了**：`infected` 來自 `clinical_severity`，所以症狀、住院、死亡等「結果端」欄位都不能當特徵（那是答案）。Task B 只有 24% 正例——這個不平衡在 Step 4 會決定我們為什麼看 AUC 不看準確率。

In [ ]:
# --- Step 1: 問題定義與目標變數 ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Task A: 預測感染
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Task B: 預測重症結局（住院或死亡）
df["severe_outcome"] = (
    (df["hospitalized"] == 1) | (df["outcome"] == "dead")
).astype(int)

print("=== 預測目標分布 ===")
print(f"Task A (infected):       {df['infected'].value_counts().to_dict()}")
print(f"Task B (severe_outcome): {df['severe_outcome'].value_counts().to_dict()}")
print(f"\nTask A 正例比例：{df['infected'].mean():.1%}")
print(f"Task B 正例比例：{df['severe_outcome'].mean():.1%}")

## Step 2 — 特徵工程：把雜亂病歷翻譯成數字

模型只會算數學，看不懂「男/女」「A 棟」，也分不清「年齡 85」和「floor 3」的尺度。三種欄位、三種處理：**數值** `age` → `StandardScaler` 標準化；**類別** `sex/wing/...` → `OneHotEncoder` 拆成 0/1 開關（避免被當成大小順序）；**二元** 共病/`shower_use` → `passthrough` 直接用。

> 🧭 **挑特徵三原則**：① 領域知識優先（入住/暴露當下就知道的線索）；② **鐵律：不用「結果之後」才有的欄位**（症狀=偷看答案，AUC 會假高到 0.99）；③ 別什麼都塞（280 筆撐不起太多特徵，會 overfit）。

In [ ]:
# --- Step 2: 特徵工程 ---
# 注意：不能用症狀（fever, cough...）當特徵！
# 因為症狀是感染「後」才出現的 → data leakage

num_cols = ["age"]  # 數值特徵

cat_cols = ["sex", "smoking_history", "functional_status", "wing"]  # 類別特徵

bin_cols = [  # 二元特徵（0/1）
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

feature_cols = num_cols + cat_cols + bin_cols
X = df[feature_cols]
y_infected = df["infected"]
y_severe = df["severe_outcome"]

print(f"特徵數：{len(feature_cols)}")
print(f"  數值：{num_cols}")
print(f"  類別：{cat_cols}")
print(f"  二元：{bin_cols}")
print(f"\n樣本數：{len(X)}")

## Step 3 — Pipeline：綁成一條、順便防洩漏

用 `ColumnTransformer` + `Pipeline` 把「前處理 + 模型」串成一個物件。

> 🔒 **為什麼不自己先 `scaler.fit(X)`？** 標準化要用平均值/標準差；若在切分前用**全部**資料算，測試集資訊就滲進訓練＝最隱蔽的 data leakage。Pipeline 保證每一折的縮放只從**該折的訓練資料**學，測試折不參與。

In [ ]:
# --- Step 3: sklearn Pipeline ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# 前處理：數值標準化 + 類別 one-hot + 二元直接通過
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

# Logistic Regression baseline
clf_lr = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])

print("Pipeline 結構：")
print(clf_lr)

## Step 4 — 交叉驗證 + AUC：到底在比什麼？

> 🔁 **交叉驗證**不是比兩個模型，是給**同一個模型**打更可信的分數：切 5 折、每筆都輪流當一次考題 → 5 個 AUC，看**平均**（多強）和**標準差**（穩不穩）。
>
> 🎯 **AUC 白話定義**：隨機抓一個感染者＋一個健康人，模型把感染者排前面的機率。0.5=閉眼猜、0.7=及格、0.8=不錯、1.0=可疑（查 leakage）。
>
> ⚠️ **為什麼不用準確率？** Task B 全猜「沒重症」就有 76% 準確率卻沒用；AUC 看排序、不受門檻影響，不平衡時誠實得多。

In [ ]:
# --- Step 4: 交叉驗證 + AUC ---
# Task A: 預測感染
scores_lr_a = cross_val_score(clf_lr, X, y_infected, cv=5, scoring="roc_auc")
print(f"=== Task A: 預測感染 ===")
print(f"Logistic Regression 5-fold CV AUC = {scores_lr_a.mean():.3f} \u00b1 {scores_lr_a.std():.3f}")

# Task B: 預測重症
scores_lr_b = cross_val_score(clf_lr, X, y_severe, cv=5, scoring="roc_auc")
print(f"\n=== Task B: 預測重症 ===")
print(f"Logistic Regression 5-fold CV AUC = {scores_lr_b.mean():.3f} \u00b1 {scores_lr_b.std():.3f}")

print("\n\u2192 AUC 0.5 = 隨機猜測，0.7+ = 可接受，0.8+ = 不錯")

## Step 5 — Random Forest：換一顆更聰明的腦袋

只把 pipeline 的 `model` 換成 Random Forest（一群決策樹投票，能抓非線性/交互作用），**其他完全沒動**——這就是包 Pipeline 的好處。

> 🌲 **但結果很誠實**：這 280 筆弱訊號資料上，RF 幾乎追平 logistic（兩者都在 0.6 上下）。不是 RF 笨，是**線索本來就薄**。ML 真正發威要等 Part B 的大資料 + 非線性場子。

In [ ]:
# --- Step 5: Random Forest 進階模型 ---
from sklearn.ensemble import RandomForestClassifier

clf_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Task A
scores_rf_a = cross_val_score(clf_rf, X, y_infected, cv=5, scoring="roc_auc")
print(f"=== Task A: 預測感染 ===")
print(f"Random Forest 5-fold CV AUC = {scores_rf_a.mean():.3f} \u00b1 {scores_rf_a.std():.3f}")

# Task B
scores_rf_b = cross_val_score(clf_rf, X, y_severe, cv=5, scoring="roc_auc")
print(f"\n=== Task B: 預測重症 ===")
print(f"Random Forest 5-fold CV AUC = {scores_rf_b.mean():.3f} \u00b1 {scores_rf_b.std():.3f}")

## Step 6 — 特徵重要性：哪條線索最有用？

> 🔍 **置換重要性**：把某一欄打亂洗牌，看 AUC 掉多少——掉越多＝越重要。就像抽掉病歷上一條線索，看實習醫師準度掉多少。
>
> 🧭 **重要 ≠ 有因果**：能幫忙預測，不代表改變它就能防病（因果是 Ch12）。值得跟 Ch06 的 adjusted OR 對照。

In [ ]:
# --- Step 6: 特徵重要性（Permutation Importance）---
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_infected, test_size=0.3, random_state=42,
)

clf_rf.fit(X_train, y_train)
perm = permutation_importance(
    clf_rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc",
)

# 排序
imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
        color="#2c7fb8", alpha=0.8)
ax.set_xlabel("Permutation Importance (AUC decrease)")
ax.set_title("Task A (infected) \u2014 Feature Importance")
plt.tight_layout()
plt.show()

print("\n=== Top 5 重要特徵 ===")
top5 = imp_df.nlargest(5, "importance")
for _, row in top5.iterrows():
    print(f"  {row['feature']:25s}  importance = {row['importance']:.4f}")

In [ ]:
# --- Step 7: 模型比較摘要 ---
from sklearn.metrics import roc_auc_score

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Task A AUC": [
        f"{scores_lr_a.mean():.3f} \u00b1 {scores_lr_a.std():.3f}",
        f"{scores_rf_a.mean():.3f} \u00b1 {scores_rf_a.std():.3f}",
    ],
    "Task B AUC": [
        f"{scores_lr_b.mean():.3f} \u00b1 {scores_lr_b.std():.3f}",
        f"{scores_rf_b.mean():.3f} \u00b1 {scores_rf_b.std():.3f}",
    ],
})
print("=== 模型比較 ===")
print(results.to_string(index=False))

print("\n\u2192 在小樣本（280 筆）中，簡單的 Logistic Regression 通常與 Random Forest 表現接近")
print("\u2192 複雜模型容易 overfit，交叉驗證的標準差可以反映這一點")
print("\u2192 ML 特徵重要性與 Ch06 邏輯斯迴歸的 adjusted OR 方向是否一致？")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 問題定義 | 區分 Task A（感染）vs Task B（重症），避免 data leakage |
| 特徵工程 | 數值 / 類別 / 二元特徵分開處理 |
| Pipeline | `ColumnTransformer` + `Pipeline` 確保前處理在 CV fold 內 |
| 交叉驗證 | `cross_val_score(cv=5, scoring='roc_auc')` |
| Random Forest | 非線性模型，但在小樣本中優勢有限 |
| 特徵重要性 | `permutation_importance` 找出預測力最強的特徵 |

**結論**：280 筆資料的 ML 模型可以作為 baseline，但預測效能受限於樣本量。
重要的是特徵重要性的排序——如果 ML 和迴歸分析指向相同的危險因子，結論更可靠。

下一章（Ch11），我們嘗試 PyTorch 深度學習——同時討論「280 筆用 DL 是否合理」。

---

**接下來 → Part B（`10_ml_advanced`）**

你可能有點失望：花了大把力氣，Random Forest 幾乎贏不了 logistic。這**不是你的錯**——是「280 筆、訊號又弱」的資料本來就難。ML 真正發威，是在**資料量大、風險非線性、有交互作用**的時候。下一個 notebook 換上更大的沙盒，帶你看模型動物園、Ensemble、SHAP 與完整評估套餐。